# 06 · Tiempos reales de transporte público

Este notebook consulta una instancia local de **OpenTripPlanner (OTP)** construida con:

- GTFS oficiales del Consorcio Regional de Transportes de Madrid.
- OpenStreetMap para accesos peatonales, transbordos y red viaria.
- Los 131 barrios oficiales de `zones_master.csv`.
- Los destinos definidos en `destinations.csv`.

## Objetivos

1. Comprobar que OpenTripPlanner está disponible.
2. Consultar rutas reales de transporte público.
3. Calcular un escenario de ida por la mañana y otro de vuelta por la tarde.
4. Conservar hasta tres itinerarios por consulta.
5. Seleccionar el primer itinerario devuelto por OTP como alternativa principal.
6. Guardar resultados, errores y auditoría.
7. Construir el tiempo diario de desplazamiento por barrio y destino.

> El notebook comienza en **modo de prueba**. No se deben ejecutar las 786 consultas definitivas hasta comprobar que el resultado de prueba es correcto.

## 1. Importaciones

In [37]:
from __future__ import annotations

import json
import time
from datetime import date, datetime, time as dt_time
from pathlib import Path
from typing import Any
from zoneinfo import ZoneInfo

import pandas as pd
import requests

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

## 2. Localización automática del proyecto

El notebook puede ejecutarse desde la raíz del proyecto o desde la carpeta `notebooks`.

In [38]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    candidates = [start, *start.parents]
    for candidate in candidates:
        zones_file = candidate / "data" / "processed" / "zones_master.csv"
        if zones_file.exists():
            return candidate

    raise FileNotFoundError(
        "No se ha encontrado la raíz del proyecto. "
        "Debe existir data/processed/zones_master.csv."
    )


PROJECT_ROOT = find_project_root()

ZONES_PATH = PROJECT_ROOT / "data" / "processed" / "zones_master.csv"
DESTINATIONS_PATH = PROJECT_ROOT / "data" / "reference" / "destinations.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

REQUEST_AUDIT_PATH = OUTPUT_DIR / "transit_route_requests_audit.csv"
ITINERARIES_PATH = OUTPUT_DIR / "transit_route_itineraries.csv"
SUMMARY_PATH = OUTPUT_DIR / "transit_routes_by_neighborhood.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Barrios: {ZONES_PATH}")
print(f"Destinos: {DESTINATIONS_PATH}")

Raíz del proyecto: C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas
Barrios: C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\processed\zones_master.csv
Destinos: C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\reference\destinations.csv


## 3. Configuración del experimento

### Escenarios

- **Mañana:** llegar al destino como máximo a las 09:00.
- **Tarde:** salir del destino a partir de las 18:00.

La fecha elegida debe estar cubierta por los calendarios GTFS descargados.

### Modo de prueba

Con `TEST_MODE = True` se consultan únicamente tres barrios.  
Cuando todas las validaciones sean correctas, se cambia a `False`.

In [39]:
OTP_ENDPOINT = "http://localhost:8080/otp/gtfs/v1"
MADRID_TZ = ZoneInfo("Europe/Madrid")

# Fecha laborable incluida en los GTFS descargados.
SERVICE_DATE = date.fromisoformat("2026-07-29")

# Mantener True durante la primera ejecución.
TEST_MODE = False

# Barrios heterogéneos para probar centro, periferia y sureste.
TEST_ZONE_KEYS = ["MAD-011", "MAD-104", "MAD-192"]

MAX_ITINERARIES = 3
REQUEST_TIMEOUT_SECONDS = 210
RETRIES = 2
CHECKPOINT_EVERY = 10
RESUME = True

SCENARIOS = [
    {
        "scenario_id": "morning_arrival_0900",
        "scenario_name": "Llegada al destino a las 09:00",
        "direction": "to_destination",
        "date_time_field": "latestArrival",
        "clock_time": dt_time(9, 0),
    },
    {
        "scenario_id": "evening_departure_1800",
        "scenario_name": "Salida del destino a las 18:00",
        "direction": "from_destination",
        "date_time_field": "earliestDeparture",
        "clock_time": dt_time(18, 0),
    },
]

print("Fecha de servicio:", SERVICE_DATE, SERVICE_DATE.strftime("%A"))
print("Modo de prueba:", TEST_MODE)

Fecha de servicio: 2026-07-29 Wednesday
Modo de prueba: False


## 4. Lectura y validación de barrios y destinos

In [40]:
zones = pd.read_csv(ZONES_PATH)
destinations = pd.read_csv(DESTINATIONS_PATH)

required_zone_columns = {
    "zone_key",
    "neighborhood_name",
    "district_name",
    "latitude",
    "longitude",
}
required_destination_columns = {
    "destination_id",
    "destination_name",
    "latitude",
    "longitude",
    "active",
}

missing_zone_columns = required_zone_columns - set(zones.columns)
missing_destination_columns = required_destination_columns - set(destinations.columns)

if missing_zone_columns:
    raise ValueError(f"Faltan columnas en zones_master.csv: {sorted(missing_zone_columns)}")

if missing_destination_columns:
    raise ValueError(
        f"Faltan columnas en destinations.csv: {sorted(missing_destination_columns)}"
    )

zones["zone_key"] = zones["zone_key"].astype(str)
zones["latitude"] = pd.to_numeric(zones["latitude"], errors="coerce")
zones["longitude"] = pd.to_numeric(zones["longitude"], errors="coerce")

destinations["destination_id"] = destinations["destination_id"].astype(str)
destinations["latitude"] = pd.to_numeric(destinations["latitude"], errors="coerce")
destinations["longitude"] = pd.to_numeric(destinations["longitude"], errors="coerce")
destinations["active"] = pd.to_numeric(destinations["active"], errors="coerce").fillna(0).astype(int)

active_destinations = destinations.loc[destinations["active"].eq(1)].copy()

if zones["zone_key"].duplicated().any():
    raise ValueError("Existen zone_key duplicados.")

if active_destinations["destination_id"].duplicated().any():
    raise ValueError("Existen destination_id duplicados.")

if zones[["latitude", "longitude"]].isna().any().any():
    raise ValueError("Hay barrios sin coordenadas.")

if active_destinations[["latitude", "longitude"]].isna().any().any():
    raise ValueError("Hay destinos activos sin coordenadas.")

if len(zones) != 131:
    raise ValueError(f"Se esperaban 131 barrios y se han encontrado {len(zones)}.")

if active_destinations.empty:
    raise ValueError("No hay destinos activos.")

print(f"Barrios: {len(zones)}")
print(f"Destinos activos: {len(active_destinations)}")

display(
    active_destinations[
        ["destination_id", "destination_name", "latitude", "longitude"]
    ]
)

Barrios: 131
Destinos activos: 3


,destination_id,destination_name,latitude,longitude
0,DEST_001,Puerta del Sol,40.416775,-3.703790
1,DEST_002,Nuevos Ministerios,40.446610,-3.692436
2,DEST_003,UC3M Campus de Leganés,40.331755,-3.765697


## 5. Comprobación de OpenTripPlanner

Docker Desktop debe estar abierto y el contenedor `otp-madrid` debe estar ejecutándose.

In [41]:
HEALTH_QUERY = '''
query HealthCheck {
  routes {
    shortName
  }
}
'''

health_response = requests.post(
    OTP_ENDPOINT,
    json={"query": HEALTH_QUERY, "operationName": "HealthCheck"},
    headers={
        "Content-Type": "application/json",
        "OTPTimeout": "180000",
        "Accept-Language": "es",
    },
    timeout=30,
)

health_response.raise_for_status()
health_payload = health_response.json()

if health_payload.get("errors"):
    raise RuntimeError(
        "OTP responde, pero GraphQL devuelve errores:\n"
        + json.dumps(health_payload["errors"], indent=2, ensure_ascii=False)
    )

route_count = len(health_payload.get("data", {}).get("routes", []))
print(f"OpenTripPlanner responde correctamente. Rutas cargadas: {route_count}")

OpenTripPlanner responde correctamente. Rutas cargadas: 704


## 6. Consulta GraphQL de planificación

Se utiliza `planConnection`, que es la consulta recomendada en las versiones actuales de OTP.

El resultado conserva:

- Duración total.
- Hora de salida y llegada.
- Tiempo andando.
- Distancia andando.
- Tiempo de espera.
- Número de transbordos.
- Modos utilizados.
- Líneas y operadores.

In [42]:
PLAN_QUERY = '''
query PlanRoute(
  $origin: PlanLabeledLocationInput!,
  $destination: PlanLabeledLocationInput!,
  $dateTime: PlanDateTimeInput!,
  $first: Int!
) {
  planConnection(
    origin: $origin,
    destination: $destination,
    dateTime: $dateTime,
    first: $first,
    modes: {
      transitOnly: true
    }
  ) {
    searchDateTime
    routingErrors {
      code
      description
      inputField
    }
    edges {
      node {
        start
        end
        duration
        generalizedCost
        waitingTime
        walkTime
        walkDistance
        numberOfTransfers
        legs {
          mode
          duration
          distance
          transitLeg
          from {
            name
          }
          to {
            name
          }
          route {
            shortName
            longName
          }
          agency {
            name
          }
        }
      }
    }
  }
}
'''

## 7. Funciones auxiliares

In [43]:
def build_location(label: str, latitude: float, longitude: float) -> dict[str, Any]:
    return {
        "label": str(label),
        "location": {
            "coordinate": {
                "latitude": float(latitude),
                "longitude": float(longitude),
            }
        },
    }


def build_scenario_datetime(
    service_date: date,
    clock_time: dt_time,
) -> str:
    value = datetime.combine(service_date, clock_time, tzinfo=MADRID_TZ)
    return value.isoformat()


def graphql_request(
    query: str,
    variables: dict[str, Any],
    operation_name: str,
    retries: int = RETRIES,
) -> dict[str, Any]:
    last_exception: Exception | None = None

    for attempt in range(retries + 1):
        try:
            response = requests.post(
                OTP_ENDPOINT,
                json={
                    "query": query,
                    "variables": variables,
                    "operationName": operation_name,
                },
                headers={
                    "Content-Type": "application/json",
                    "OTPTimeout": "180000",
                    "Accept-Language": "es",
                },
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            response.raise_for_status()
            payload = response.json()

            if payload.get("errors"):
                raise RuntimeError(
                    json.dumps(payload["errors"], ensure_ascii=False)
                )

            return payload

        except (requests.RequestException, ValueError, RuntimeError) as exc:
            last_exception = exc

            if attempt < retries:
                wait_seconds = 2 ** attempt
                print(
                    f"Reintento {attempt + 1}/{retries} "
                    f"tras error: {exc}"
                )
                time.sleep(wait_seconds)

    raise RuntimeError(f"Consulta OTP fallida: {last_exception}")

In [44]:
def parse_itinerary(
    itinerary: dict[str, Any],
    base_record: dict[str, Any],
    rank: int,
) -> dict[str, Any]:
    legs = itinerary.get("legs") or []

    all_modes = [
        leg.get("mode")
        for leg in legs
        if leg.get("mode")
    ]
    transit_modes = [
        leg.get("mode")
        for leg in legs
        if leg.get("transitLeg") and leg.get("mode")
    ]

    routes_used = []
    agencies_used = []

    for leg in legs:
        route = leg.get("route") or {}
        agency = leg.get("agency") or {}

        route_label = route.get("shortName") or route.get("longName")
        if route_label:
            routes_used.append(str(route_label))

        agency_name = agency.get("name")
        if agency_name:
            agencies_used.append(str(agency_name))

    return {
        **base_record,
        "itinerary_rank": rank,
        "start_datetime": itinerary.get("start"),
        "end_datetime": itinerary.get("end"),
        "duration_minutes": (
            itinerary.get("duration") / 60
            if itinerary.get("duration") is not None
            else None
        ),
        "walk_time_minutes": (
            itinerary.get("walkTime") / 60
            if itinerary.get("walkTime") is not None
            else None
        ),
        "waiting_time_minutes": (
            itinerary.get("waitingTime") / 60
            if itinerary.get("waitingTime") is not None
            else None
        ),
        "walk_distance_meters": itinerary.get("walkDistance"),
        "number_of_transfers": itinerary.get("numberOfTransfers"),
        "generalized_cost": itinerary.get("generalizedCost"),
        "leg_count": len(legs),
        "transit_leg_count": sum(
            bool(leg.get("transitLeg"))
            for leg in legs
        ),
        "all_modes": "|".join(dict.fromkeys(all_modes)),
        "transit_modes": "|".join(dict.fromkeys(transit_modes)),
        "routes_used": "|".join(dict.fromkeys(routes_used)),
        "agencies_used": "|".join(dict.fromkeys(agencies_used)),
        "legs_json": json.dumps(
            legs,
            ensure_ascii=False,
            separators=(",", ":"),
        ),
    }

## 8. Prueba de una única ruta

Antes de automatizar todas las consultas, se prueba:

- **Origen:** Palacio.
- **Destino:** Puerta del Sol.
- **Escenario:** llegada a las 09:00.

In [45]:
test_zone = zones.loc[
    zones["zone_key"].eq("MAD-192")
].iloc[0]

test_destination = active_destinations.loc[
    active_destinations["destination_id"].eq("DEST_002")
].iloc[0]

test_scenario = SCENARIOS[0]

test_datetime = build_scenario_datetime(
    SERVICE_DATE,
    test_scenario["clock_time"],
)

test_variables = {
    "origin": build_location(
        test_zone["neighborhood_name"],
        test_zone["latitude"],
        test_zone["longitude"],
    ),
    "destination": build_location(
        test_destination["destination_name"],
        test_destination["latitude"],
        test_destination["longitude"],
    ),
    "dateTime": {
        test_scenario["date_time_field"]: test_datetime,
    },
    "first": MAX_ITINERARIES,
}

test_payload = graphql_request(
    PLAN_QUERY,
    test_variables,
    "PlanRoute",
)

test_plan = test_payload["data"]["planConnection"]
test_edges = test_plan.get("edges") or []
test_routing_errors = test_plan.get("routingErrors") or []

print("Fecha/hora de consulta:", test_datetime)
print("Itinerarios encontrados:", len(test_edges))
print("Errores de enrutamiento:", test_routing_errors)

if not test_edges:
    raise RuntimeError(
        "OTP no ha encontrado itinerarios para la prueba. "
        "Comprueba la fecha de servicio y los GTFS."
    )

Fecha/hora de consulta: 2026-07-29T09:00:00+02:00
Itinerarios encontrados: 3
Errores de enrutamiento: []


In [46]:
test_rows = []

test_base = {
    "request_key": (
        f"{test_zone['zone_key']}|"
        f"{test_destination['destination_id']}|"
        f"{test_scenario['scenario_id']}"
    ),
    "zone_key": test_zone["zone_key"],
    "neighborhood_name": test_zone["neighborhood_name"],
    "district_name": test_zone["district_name"],
    "destination_id": test_destination["destination_id"],
    "destination_name": test_destination["destination_name"],
    "scenario_id": test_scenario["scenario_id"],
    "scenario_name": test_scenario["scenario_name"],
    "direction": test_scenario["direction"],
    "requested_datetime": test_datetime,
}

for rank, edge in enumerate(test_edges, start=1):
    test_rows.append(
        parse_itinerary(
            edge["node"],
            test_base,
            rank,
        )
    )

test_results = pd.DataFrame(test_rows)

display(
    test_results[
        [
            "itinerary_rank",
            "start_datetime",
            "end_datetime",
            "duration_minutes",
            "walk_time_minutes",
            "waiting_time_minutes",
            "number_of_transfers",
            "transit_modes",
            "routes_used",
        ]
    ]
)

,itinerary_rank,start_datetime,end_datetime,duration_minutes,walk_time_minutes,waiting_time_minutes,number_of_transfers,transit_modes,routes_used
0,1,2026-07-29T07:43:00+02:00,2026-07-29T08:59:40+02:00,76.666667,7.883333,15.0,1,BUS,E4|C2
1,2,2026-07-29T07:43:00+02:00,2026-07-29T08:59:29+02:00,76.483333,9.233333,15.0,1,BUS,E4|C2
2,3,2026-07-29T07:42:00+02:00,2026-07-29T08:58:40+02:00,76.666667,7.883333,15.0,1,BUS,E4|C2


## 9. Selección de barrios para la ejecución

En modo de prueba se utilizan tres barrios.  
En modo definitivo se utilizan los 131.

In [47]:
if TEST_MODE:
    missing_test_keys = sorted(set(TEST_ZONE_KEYS) - set(zones["zone_key"]))
    if missing_test_keys:
        raise ValueError(
            f"Los siguientes barrios de prueba no existen: {missing_test_keys}"
        )

    zones_to_process = (
        zones.loc[zones["zone_key"].isin(TEST_ZONE_KEYS)]
        .sort_values("zone_key")
        .copy()
    )
else:
    zones_to_process = zones.sort_values("zone_key").copy()

expected_requests = (
    len(zones_to_process)
    * len(active_destinations)
    * len(SCENARIOS)
)

print(f"Barrios a procesar: {len(zones_to_process)}")
print(f"Destinos: {len(active_destinations)}")
print(f"Escenarios: {len(SCENARIOS)}")
print(f"Consultas previstas: {expected_requests}")

display(
    zones_to_process[
        ["zone_key", "neighborhood_name", "district_name"]
    ]
)

Barrios a procesar: 131
Destinos: 3
Escenarios: 2
Consultas previstas: 786


,zone_key,neighborhood_name,district_name
0,MAD-011,Palacio,Centro
1,MAD-012,Embajadores,Centro
2,MAD-013,Cortes,Centro
3,MAD-014,Justicia,Centro
4,MAD-015,Universidad,Centro
...,...,...,...
126,MAD-211,Alameda de Osuna,Barajas
127,MAD-212,Aeropuerto,Barajas
128,MAD-213,Casco Histórico de Barajas,Barajas
129,MAD-214,Timón,Barajas


## 10. Recuperación de progreso anterior

Si el notebook se interrumpe, puede continuar sin repetir las consultas ya completadas correctamente.

In [62]:
if RESUME and REQUEST_AUDIT_PATH.exists():
    existing_audit = pd.read_csv(REQUEST_AUDIT_PATH)
else:
    existing_audit = pd.DataFrame()

if RESUME and ITINERARIES_PATH.exists():
    existing_itineraries = pd.read_csv(ITINERARIES_PATH)
else:
    existing_itineraries = pd.DataFrame()

completed_keys: set[str] = set()

if not existing_audit.empty and {"request_key", "request_status"}.issubset(
    existing_audit.columns
):
    completed_keys = set(
        existing_audit.loc[
            existing_audit["request_status"].isin(["success", "no_route"]),
            "request_key",
        ].astype(str)
    )

print(f"Consultas ya completadas y reutilizables: {len(completed_keys)}")

Consultas ya completadas y reutilizables: 786


## 11. Ejecución automatizada

In [63]:
new_audit_records: list[dict[str, Any]] = []
new_itinerary_records: list[dict[str, Any]] = []

total_counter = 0
executed_counter = 0

for _, zone in zones_to_process.iterrows():
    for _, destination in active_destinations.iterrows():
        for scenario in SCENARIOS:
            total_counter += 1

            request_key = (
                f"{zone['zone_key']}|"
                f"{destination['destination_id']}|"
                f"{scenario['scenario_id']}"
            )

            if request_key in completed_keys:
                continue

            requested_datetime = build_scenario_datetime(
                SERVICE_DATE,
                scenario["clock_time"],
            )

            if scenario["direction"] == "to_destination":
                origin = build_location(
                    zone["neighborhood_name"],
                    zone["latitude"],
                    zone["longitude"],
                )
                target = build_location(
                    destination["destination_name"],
                    destination["latitude"],
                    destination["longitude"],
                )
            elif scenario["direction"] == "from_destination":
                origin = build_location(
                    destination["destination_name"],
                    destination["latitude"],
                    destination["longitude"],
                )
                target = build_location(
                    zone["neighborhood_name"],
                    zone["latitude"],
                    zone["longitude"],
                )
            else:
                raise ValueError(
                    f"Dirección desconocida: {scenario['direction']}"
                )

            variables = {
                "origin": origin,
                "destination": target,
                "dateTime": {
                    scenario["date_time_field"]: requested_datetime,
                },
                "first": MAX_ITINERARIES,
            }

            base_record = {
                "request_key": request_key,
                "zone_key": zone["zone_key"],
                "neighborhood_name": zone["neighborhood_name"],
                "district_name": zone["district_name"],
                "origin_latitude": origin["location"]["coordinate"]["latitude"],
                "origin_longitude": origin["location"]["coordinate"]["longitude"],
                "destination_id": destination["destination_id"],
                "destination_name": destination["destination_name"],
                "target_latitude": target["location"]["coordinate"]["latitude"],
                "target_longitude": target["location"]["coordinate"]["longitude"],
                "scenario_id": scenario["scenario_id"],
                "scenario_name": scenario["scenario_name"],
                "direction": scenario["direction"],
                "service_date": SERVICE_DATE.isoformat(),
                "requested_datetime": requested_datetime,
                "calculation_timestamp": datetime.now(MADRID_TZ).isoformat(),
            }

            started_at = time.perf_counter()

            try:
                payload = graphql_request(
                    PLAN_QUERY,
                    variables,
                    "PlanRoute",
                )

                plan = payload["data"]["planConnection"]
                edges = plan.get("edges") or []
                routing_errors = plan.get("routingErrors") or []

                elapsed_seconds = time.perf_counter() - started_at

                if edges:
                    request_status = "success"

                    for rank, edge in enumerate(edges, start=1):
                        new_itinerary_records.append(
                            parse_itinerary(
                                edge["node"],
                                base_record,
                                rank,
                            )
                        )
                else:
                    request_status = "no_route"

                new_audit_records.append(
                    {
                        **base_record,
                        "request_status": request_status,
                        "itineraries_found": len(edges),
                        "routing_errors_json": json.dumps(
                            routing_errors,
                            ensure_ascii=False,
                            separators=(",", ":"),
                        ),
                        "request_error": None,
                        "elapsed_seconds": elapsed_seconds,
                    }
                )

            except Exception as exc:
                elapsed_seconds = time.perf_counter() - started_at

                new_audit_records.append(
                    {
                        **base_record,
                        "request_status": "error",
                        "itineraries_found": 0,
                        "routing_errors_json": "[]",
                        "request_error": str(exc),
                        "elapsed_seconds": elapsed_seconds,
                    }
                )

            executed_counter += 1

            if (
                executed_counter % CHECKPOINT_EVERY == 0
                or executed_counter == expected_requests
            ):
                combined_audit = pd.concat(
                    [
                        existing_audit,
                        pd.DataFrame(new_audit_records),
                    ],
                    ignore_index=True,
                ).drop_duplicates(
                    subset=["request_key"],
                    keep="last",
                )

                combined_itineraries = pd.concat(
                    [
                        existing_itineraries,
                        pd.DataFrame(new_itinerary_records),
                    ],
                    ignore_index=True,
                )

                if not combined_itineraries.empty:
                    combined_itineraries = combined_itineraries.drop_duplicates(
                        subset=["request_key", "itinerary_rank"],
                        keep="last",
                    )

                combined_audit.to_csv(
                    REQUEST_AUDIT_PATH,
                    index=False,
                    encoding="utf-8-sig",
                )

                combined_itineraries.to_csv(
                    ITINERARIES_PATH,
                    index=False,
                    encoding="utf-8-sig",
                )

                print(
                    f"Progreso guardado: "
                    f"{total_counter}/{expected_requests} combinaciones; "
                    f"{executed_counter} nuevas consultas."
                )

# Guardado final obligatorio, aunque el número de consultas nuevas
# no sea múltiplo de CHECKPOINT_EVERY.
combined_audit = pd.concat(
    [
        existing_audit,
        pd.DataFrame(new_audit_records),
    ],
    ignore_index=True,
).drop_duplicates(
    subset=["request_key"],
    keep="last",
)

combined_itineraries = pd.concat(
    [
        existing_itineraries,
        pd.DataFrame(new_itinerary_records),
    ],
    ignore_index=True,
)

if not combined_itineraries.empty:
    combined_itineraries = combined_itineraries.drop_duplicates(
        subset=["request_key", "itinerary_rank"],
        keep="last",
    )

combined_audit.to_csv(
    REQUEST_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

combined_itineraries.to_csv(
    ITINERARIES_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(
    f"Guardado final completado: "
    f"{len(combined_audit)} consultas auditadas y "
    f"{len(combined_itineraries)} itinerarios."
)

print("Ejecución terminada.")

Guardado final completado: 786 consultas auditadas y 2239 itinerarios.
Ejecución terminada.


## 12. Lectura de resultados y auditoría

In [64]:
request_audit = pd.read_csv(REQUEST_AUDIT_PATH)
itineraries = pd.read_csv(ITINERARIES_PATH)

current_zone_keys = set(zones_to_process["zone_key"])
current_destination_ids = set(active_destinations["destination_id"])
current_scenarios = {scenario["scenario_id"] for scenario in SCENARIOS}

current_audit = request_audit.loc[
    request_audit["zone_key"].isin(current_zone_keys)
    & request_audit["destination_id"].isin(current_destination_ids)
    & request_audit["scenario_id"].isin(current_scenarios)
].copy()

current_itineraries = itineraries.loc[
    itineraries["zone_key"].isin(current_zone_keys)
    & itineraries["destination_id"].isin(current_destination_ids)
    & itineraries["scenario_id"].isin(current_scenarios)
].copy()

print("Estado de las consultas:")
display(
    current_audit["request_status"]
    .value_counts(dropna=False)
    .rename_axis("request_status")
    .reset_index(name="count")
)

print("Itinerarios almacenados:", len(current_itineraries))

Estado de las consultas:


,request_status,count
0,success,749
1,no_route,37


Itinerarios almacenados: 2239


In [71]:
error_requests = current_audit.loc[
    current_audit["request_status"].eq("error"),
    [
        "zone_key",
        "neighborhood_name",
        "destination_name",
        "scenario_id",
        "request_error",
    ],
]

no_route_requests = current_audit.loc[
    current_audit["request_status"].eq("no_route"),
    [
        "zone_key",
        "neighborhood_name",
        "destination_name",
        "scenario_id",
        "routing_errors_json",
    ],
]

print(f"Consultas con error técnico: {len(error_requests)}")
print(f"Consultas sin ruta: {len(no_route_requests)}")

if not error_requests.empty:
    display(error_requests.head(20))

if not no_route_requests.empty:
    display(no_route_requests.head(20))

Consultas con error técnico: 0
Consultas sin ruta: 37


,zone_key,neighborhood_name,destination_name,scenario_id,routing_errors_json
0,MAD-011,Palacio,Puerta del Sol,morning_arrival_0900,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
1,MAD-011,Palacio,Puerta del Sol,evening_departure_1800,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
18,MAD-012,Embajadores,Puerta del Sol,morning_arrival_0900,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
19,MAD-012,Embajadores,Puerta del Sol,evening_departure_1800,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
24,MAD-013,Cortes,Puerta del Sol,morning_arrival_0900,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
25,MAD-013,Cortes,Puerta del Sol,evening_departure_1800,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
30,MAD-014,Justicia,Puerta del Sol,morning_arrival_0900,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
31,MAD-014,Justicia,Puerta del Sol,evening_departure_1800,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
36,MAD-015,Universidad,Puerta del Sol,morning_arrival_0900,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
37,MAD-015,Universidad,Puerta del Sol,evening_departure_1800,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."


In [72]:
import json
from collections import Counter

no_route_audit = current_audit.loc[
    current_audit["request_status"].eq("no_route")
].copy()

def obtener_codigos_error(valor):
    try:
        errores = json.loads(valor)
    except (TypeError, json.JSONDecodeError):
        return ["ERROR_DE_LECTURA"]

    if not errores:
        return ["SIN_CODIGO"]

    return [
        error.get("code", "DESCONOCIDO")
        for error in errores
    ]

contador_codigos = Counter()

for valor in no_route_audit["routing_errors_json"]:
    contador_codigos.update(
        obtener_codigos_error(valor)
    )

resumen_no_route = pd.DataFrame(
    contador_codigos.items(),
    columns=["codigo", "numero_consultas"],
).sort_values(
    "numero_consultas",
    ascending=False,
)

display(resumen_no_route)

,codigo,numero_consultas
0,WALKING_BETTER_THAN_TRANSIT,23
1,NO_STOPS_IN_RANGE,12
2,NO_TRANSIT_CONNECTION_IN_SEARCH_WINDOW,2


In [73]:
display(
    no_route_audit[
        [
            "zone_key",
            "neighborhood_name",
            "destination_name",
            "scenario_id",
            "routing_errors_json",
        ]
    ].sort_values(
        [
            "destination_name",
            "neighborhood_name",
            "scenario_id",
        ]
    )
)

,zone_key,neighborhood_name,destination_name,scenario_id,routing_errors_json
177,MAD-053,Ciudad Jardín,Nuevos Ministerios,evening_departure_1800,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
176,MAD-053,Ciudad Jardín,Nuevos Ministerios,morning_arrival_0900,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
603,MAD-159,Costillares,Nuevos Ministerios,evening_departure_1800,"[{""code"":""NO_STOPS_IN_RANGE"",""description"":""The location was found, but no stops could be found within the search ra..."
602,MAD-159,Costillares,Nuevos Ministerios,morning_arrival_0900,"[{""code"":""NO_STOPS_IN_RANGE"",""description"":""The location was found, but no stops could be found within the search ra..."
207,MAD-062,Cuatro Caminos,Nuevos Ministerios,evening_departure_1800,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
206,MAD-062,Cuatro Caminos,Nuevos Ministerios,morning_arrival_0900,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
273,MAD-081,El Pardo,Nuevos Ministerios,evening_departure_1800,"[{""code"":""NO_STOPS_IN_RANGE"",""description"":""The location was found, but no stops could be found within the search ra..."
272,MAD-081,El Pardo,Nuevos Ministerios,morning_arrival_0900,"[{""code"":""NO_STOPS_IN_RANGE"",""description"":""The location was found, but no stops could be found within the search ra..."
165,MAD-051,El Viso,Nuevos Ministerios,evening_departure_1800,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
164,MAD-051,El Viso,Nuevos Ministerios,morning_arrival_0900,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."


In [74]:
import json
from collections import Counter

no_route_audit = current_audit.loc[
    current_audit["request_status"].eq("no_route")
].copy()

def extract_routing_codes(value):
    try:
        errors = json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return ["UNPARSEABLE"]

    if not errors:
        return ["NO_ROUTING_CODE"]

    return [
        error.get("code", "UNKNOWN")
        for error in errors
    ]

routing_code_counter = Counter()

for value in no_route_audit["routing_errors_json"]:
    routing_code_counter.update(
        extract_routing_codes(value)
    )

routing_code_summary = pd.DataFrame(
    routing_code_counter.items(),
    columns=["routing_error_code", "count"],
).sort_values(
    "count",
    ascending=False,
)

display(routing_code_summary)

,routing_error_code,count
0,WALKING_BETTER_THAN_TRANSIT,23
1,NO_STOPS_IN_RANGE,12
2,NO_TRANSIT_CONNECTION_IN_SEARCH_WINDOW,2


In [75]:
display(
    no_route_audit[
        [
            "zone_key",
            "neighborhood_name",
            "destination_name",
            "scenario_id",
            "routing_errors_json",
        ]
    ].sort_values(
        [
            "destination_name",
            "neighborhood_name",
            "scenario_id",
        ]
    )
)

,zone_key,neighborhood_name,destination_name,scenario_id,routing_errors_json
177,MAD-053,Ciudad Jardín,Nuevos Ministerios,evening_departure_1800,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
176,MAD-053,Ciudad Jardín,Nuevos Ministerios,morning_arrival_0900,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
603,MAD-159,Costillares,Nuevos Ministerios,evening_departure_1800,"[{""code"":""NO_STOPS_IN_RANGE"",""description"":""The location was found, but no stops could be found within the search ra..."
602,MAD-159,Costillares,Nuevos Ministerios,morning_arrival_0900,"[{""code"":""NO_STOPS_IN_RANGE"",""description"":""The location was found, but no stops could be found within the search ra..."
207,MAD-062,Cuatro Caminos,Nuevos Ministerios,evening_departure_1800,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
206,MAD-062,Cuatro Caminos,Nuevos Ministerios,morning_arrival_0900,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
273,MAD-081,El Pardo,Nuevos Ministerios,evening_departure_1800,"[{""code"":""NO_STOPS_IN_RANGE"",""description"":""The location was found, but no stops could be found within the search ra..."
272,MAD-081,El Pardo,Nuevos Ministerios,morning_arrival_0900,"[{""code"":""NO_STOPS_IN_RANGE"",""description"":""The location was found, but no stops could be found within the search ra..."
165,MAD-051,El Viso,Nuevos Ministerios,evening_departure_1800,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."
164,MAD-051,El Viso,Nuevos Ministerios,morning_arrival_0900,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""description"":""ES-los Origin is within a trivial distance of the destination...."


## 13. Construcción del resumen principal

Se selecciona `itinerary_rank = 1`, es decir, el primer itinerario devuelto por OTP.

No se elimina el resto: las alternativas permanecen en `transit_route_itineraries.csv`.

In [76]:
best_itineraries = (
    current_itineraries.loc[
        current_itineraries["itinerary_rank"].eq(1)
    ]
    .copy()
)

best_columns = [
    "zone_key",
    "neighborhood_name",
    "district_name",
    "destination_id",
    "destination_name",
    "scenario_id",
    "service_date",
    "requested_datetime",
    "start_datetime",
    "end_datetime",
    "duration_minutes",
    "walk_time_minutes",
    "waiting_time_minutes",
    "walk_distance_meters",
    "number_of_transfers",
    "transit_modes",
    "routes_used",
    "agencies_used",
]

best_itineraries = best_itineraries[best_columns]

if best_itineraries.duplicated(
    subset=["zone_key", "destination_id", "scenario_id"]
).any():
    raise ValueError("Hay más de un itinerario principal por consulta.")

display(best_itineraries.head())

,zone_key,neighborhood_name,district_name,destination_id,destination_name,scenario_id,service_date,requested_datetime,start_datetime,end_datetime,duration_minutes,walk_time_minutes,waiting_time_minutes,walk_distance_meters,number_of_transfers,transit_modes,routes_used,agencies_used
0,MAD-011,Palacio,Centro,DEST_002,Nuevos Ministerios,morning_arrival_0900,2026-07-29,2026-07-29T09:00:00+02:00,2026-07-29T07:51:00+02:00,2026-07-29T08:51:33+02:00,60.550000,10.633333,13.000000,707.85,1,BUS,C1,Información oficial: Consorcio Regional de Transportes de Madrid
3,MAD-011,Palacio,Centro,DEST_002,Nuevos Ministerios,evening_departure_1800,2026-07-29,2026-07-29T18:00:00+02:00,2026-07-29T18:00:00+02:00,2026-07-29T18:51:57+02:00,51.950000,21.350000,7.000000,1545.31,0,BUS,147,Información oficial: Consorcio Regional de Transportes de Madrid
6,MAD-011,Palacio,Centro,DEST_003,UC3M Campus de Leganés,morning_arrival_0900,2026-07-29,2026-07-29T09:00:00+02:00,2026-07-29T07:38:00+02:00,2026-07-29T08:56:51+02:00,78.850000,32.650000,11.066667,2306.94,1,BUS,39|482,Información oficial: Consorcio Regional de Transportes de Madrid|Consorcio Regional de Transportes de Madrid
9,MAD-011,Palacio,Centro,DEST_003,UC3M Campus de Leganés,evening_departure_1800,2026-07-29,2026-07-29T18:00:00+02:00,2026-07-29T18:00:58+02:00,2026-07-29T19:23:49+02:00,82.850000,17.266667,12.000000,1257.27,1,BUS,493|138,Consorcio Regional de Transportes de Madrid|Información oficial: Consorcio Regional de Transportes de Madrid
12,MAD-104,Aluche,Latina,DEST_001,Puerta del Sol,morning_arrival_0900,2026-07-29,2026-07-29T09:00:00+02:00,2026-07-29T08:06:00+02:00,2026-07-29T08:59:26+02:00,53.433333,15.250000,7.000000,1089.86,0,BUS,31,Información oficial: Consorcio Regional de Transportes de Madrid


## 14. Tiempo diario de desplazamiento

Se suman:

- Duración de ida por la mañana.
- Duración de vuelta por la tarde.

In [77]:
value_columns = [
    "duration_minutes",
    "walk_time_minutes",
    "waiting_time_minutes",
    "walk_distance_meters",
    "number_of_transfers",
]

daily_parts = []

for scenario_id, prefix in [
    ("morning_arrival_0900", "morning"),
    ("evening_departure_1800", "evening"),
]:
    part = best_itineraries.loc[
        best_itineraries["scenario_id"].eq(scenario_id)
    ].copy()

    rename_map = {
        column: f"{prefix}_{column}"
        for column in value_columns
    }
    rename_map.update(
        {
            "start_datetime": f"{prefix}_start_datetime",
            "end_datetime": f"{prefix}_end_datetime",
            "transit_modes": f"{prefix}_transit_modes",
            "routes_used": f"{prefix}_routes_used",
            "agencies_used": f"{prefix}_agencies_used",
        }
    )

    keep_columns = [
        "zone_key",
        "neighborhood_name",
        "district_name",
        "destination_id",
        "destination_name",
        "service_date",
        "start_datetime",
        "end_datetime",
        "transit_modes",
        "routes_used",
        "agencies_used",
        *value_columns,
    ]

    part = part[keep_columns].rename(columns=rename_map)
    daily_parts.append(part)

daily_summary = daily_parts[0].merge(
    daily_parts[1],
    on=[
        "zone_key",
        "neighborhood_name",
        "district_name",
        "destination_id",
        "destination_name",
        "service_date",
    ],
    how="outer",
    validate="one_to_one",
)

daily_summary["commute_daily_minutes"] = (
    daily_summary["morning_duration_minutes"]
    + daily_summary["evening_duration_minutes"]
)

daily_summary["commute_daily_walk_minutes"] = (
    daily_summary["morning_walk_time_minutes"]
    + daily_summary["evening_walk_time_minutes"]
)

daily_summary["commute_daily_waiting_minutes"] = (
    daily_summary["morning_waiting_time_minutes"]
    + daily_summary["evening_waiting_time_minutes"]
)

daily_summary["commute_daily_transfers"] = (
    daily_summary["morning_number_of_transfers"]
    + daily_summary["evening_number_of_transfers"]
)

daily_summary.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(f"Resumen guardado en: {SUMMARY_PATH}")
display(
    daily_summary[
        [
            "zone_key",
            "neighborhood_name",
            "destination_name",
            "morning_duration_minutes",
            "evening_duration_minutes",
            "commute_daily_minutes",
        ]
    ].head(20)
)

Resumen guardado en: C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\processed\transit_routes_by_neighborhood.csv


,zone_key,neighborhood_name,destination_name,morning_duration_minutes,evening_duration_minutes,commute_daily_minutes
0,MAD-011,Palacio,Nuevos Ministerios,60.550000,51.950000,112.500000
1,MAD-011,Palacio,UC3M Campus de Leganés,78.850000,82.850000,161.700000
2,MAD-012,Embajadores,Nuevos Ministerios,51.433333,45.216667,96.650000
3,MAD-012,Embajadores,UC3M Campus de Leganés,70.850000,75.666667,146.516667
4,MAD-013,Cortes,Nuevos Ministerios,43.350000,35.350000,78.700000
5,MAD-013,Cortes,UC3M Campus de Leganés,74.850000,80.666667,155.516667
6,MAD-014,Justicia,Nuevos Ministerios,37.750000,31.850000,69.600000
7,MAD-014,Justicia,UC3M Campus de Leganés,80.850000,84.216667,165.066667
8,MAD-015,Universidad,Nuevos Ministerios,34.983333,31.233333,66.216667
9,MAD-015,Universidad,UC3M Campus de Leganés,81.850000,90.933333,172.783333


## 15. Validaciones finales

In [78]:
expected_request_keys = {
    f"{zone_key}|{destination_id}|{scenario_id}"
    for zone_key in zones_to_process["zone_key"]
    for destination_id in active_destinations["destination_id"]
    for scenario_id in current_scenarios
}

observed_request_keys = set(current_audit["request_key"].astype(str))

missing_request_keys = sorted(
    expected_request_keys - observed_request_keys
)

duplicate_audit_keys = current_audit["request_key"].duplicated().sum()

print(f"Consultas esperadas: {len(expected_request_keys)}")
print(f"Consultas auditadas: {len(observed_request_keys)}")
print(f"Consultas ausentes: {len(missing_request_keys)}")
print(f"Claves duplicadas en auditoría: {duplicate_audit_keys}")
print(f"Filas del resumen diario: {len(daily_summary)}")

if missing_request_keys:
    print("Primeras consultas ausentes:")
    print(missing_request_keys[:10])

if duplicate_audit_keys:
    raise ValueError("La auditoría contiene request_key duplicadas.")

successful_requests = current_audit["request_status"].eq("success").sum()
failed_requests = current_audit["request_status"].eq("error").sum()
no_route_count = current_audit["request_status"].eq("no_route").sum()

print(f"Consultas correctas: {successful_requests}")
print(f"Errores técnicos: {failed_requests}")
print(f"Sin ruta: {no_route_count}")

if TEST_MODE:
    print(
        "\nPRUEBA COMPLETADA. "
        "Revisa los resultados. Después cambia TEST_MODE = False, "
        "reinicia el kernel y ejecuta todo el notebook."
    )
else:
    if len(missing_request_keys) == 0 and failed_requests == 0:
        print("\nTodas las consultas definitivas se han completado correctamente.")
    else:
        print(
            "\nLa ejecución definitiva ha terminado, "
            "pero existen consultas pendientes o con error."
        )

Consultas esperadas: 786
Consultas auditadas: 786
Consultas ausentes: 0
Claves duplicadas en auditoría: 0
Filas del resumen diario: 376
Consultas correctas: 749
Errores técnicos: 0
Sin ruta: 37

Todas las consultas definitivas se han completado correctamente.


## 16. Archivos generados

### `transit_route_requests_audit.csv`

Una fila por solicitud a OTP. Permite conocer:

- Consultas correctas.
- Consultas sin ruta.
- Errores técnicos.
- Duración de cada petición.
- Fecha y hora de cálculo.

### `transit_route_itineraries.csv`

Una fila por itinerario. Conserva hasta tres alternativas por consulta.

### `transit_routes_by_neighborhood.csv`

Una fila por combinación barrio–destino, con:

- Ida de mañana.
- Vuelta de tarde.
- Tiempo diario total.
- Caminata.
- Espera.
- Transbordos.
- Modos y líneas.

Este último archivo será la entrada principal del notebook 07.

## 17. Ejecución definitiva

Cuando el modo de prueba sea correcto:

1. Cambiar `TEST_MODE = True` por `TEST_MODE = False`.
2. Reiniciar el kernel.
3. Ejecutar todas las celdas.
4. Mantener Docker y el contenedor `otp-madrid` activos.
5. No borrar los archivos parciales: permiten reanudar la ejecución.